In [ ]:
import pandas as pd

# Load data
deliveries = pd.read_csv('../data/deliveries.csv')
matches = pd.read_csv('../data/matches.csv')

print("Deliveries shape:", deliveries.shape)
print("Deliveries columns:", deliveries.columns.tolist())
print("\nMatches shape:", matches.shape)
print("Matches columns:", matches.columns.tolist())

# Define target variable (winner for matches data)
TARGET = 'winner'

# Select relevant columns and prepare data
DROP = ['id', 'season', 'city', 'date', 'toss_winner', 'toss_decision', 'dl_applied', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']

# Prepare features and target
X = matches.drop(columns=[col for col in DROP + [TARGET] if col in matches.columns])
y = matches[TARGET]

print("\nFeatures shape:", X.shape)
print("Target shape:", y.shape)

Deliveries shape: (150460, 21)
Deliveries columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']

Matches shape: (636, 18)
Matches columns: ['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']

Features shape: (636, 5)
Target shape: (636,)


In [5]:
data = deliveries.merge(matches, left_on='match_id', right_on='id')

print(data.shape)
data.head()

(150460, 39)


,match_id,inning,batting_team,bowling_team,over,ball,batsman,non_striker,bowler,is_super_over,...,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,1,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,2,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
2,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,3,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
3,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,4,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
4,1,1,Sunrisers Hyderabad,Royal Challengers Bangalore,1,5,DA Warner,S Dhawan,TS Mills,0,...,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN


In [6]:
over_data = data.groupby(['match_id', 'inning', 'over']).agg({
    'total_runs': 'sum',
    'extra_runs': 'sum',
    'is_wicket': 'sum'
}).reset_index()
data['is_wicket'] = data['player_dismissed'].notnull().astype(int)

KeyError: "Label(s) ['is_wicket'] do not exist"

In [ ]:
print(over_data.shape)
over_data.head()


(24388, 6)


,match_id,inning,over,total_runs,extra_runs,is_wicket
0,1,1,1,7,3,0
1,1,1,2,16,1,1
2,1,1,3,6,0,0
3,1,1,4,4,0,0
4,1,1,5,9,0,0


In [ ]:
over_data['cum_runs'] = over_data.groupby(['match_id', 'inning'])['total_runs'].cumsum()

over_data['cum_wickets'] = over_data.groupby(['match_id', 'inning'])['is_wicket'].cumsum()

over_data.head()

,match_id,inning,over,total_runs,extra_runs,is_wicket,cum_runs,cum_wickets
0,1,1,1,7,3,0,7,0
1,1,1,2,16,1,1,23,1
2,1,1,3,6,0,0,29,1
3,1,1,4,4,0,0,33,1
4,1,1,5,9,0,0,42,1


In [ ]:
team_data = matches[['id', 'team1', 'winner']]

over_data = over_data.merge(team_data, left_on='match_id', right_on='id')

over_data.head()

,match_id,inning,over,total_runs,extra_runs,is_wicket,cum_runs,cum_wickets,id,team1,winner
0,1,1,1,7,3,0,7,0,1,Sunrisers Hyderabad,Sunrisers Hyderabad
1,1,1,2,16,1,1,23,1,1,Sunrisers Hyderabad,Sunrisers Hyderabad
2,1,1,3,6,0,0,29,1,1,Sunrisers Hyderabad,Sunrisers Hyderabad
3,1,1,4,4,0,0,33,1,1,Sunrisers Hyderabad,Sunrisers Hyderabad
4,1,1,5,9,0,0,42,1,1,Sunrisers Hyderabad,Sunrisers Hyderabad


In [ ]:
over_data['team1_won'] = (over_data['winner'] == over_data['team1']).astype(int)

In [ ]:
over_data = over_data.drop(columns=['id', 'team1', 'winner'])

In [ ]:
over_data.head()

,match_id,inning,over,total_runs,extra_runs,is_wicket,cum_runs,cum_wickets,team1_won
0,1,1,1,7,3,0,7,0,1
1,1,1,2,16,1,1,23,1,1
2,1,1,3,6,0,0,29,1,1
3,1,1,4,4,0,0,33,1,1
4,1,1,5,9,0,0,42,1,1


In [ ]:
X = over_data.drop(columns=['team1_won'])
y = over_data['team1_won']

print(X.shape)
print(y.shape)

(24388, 8)
(24388,)


In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(X_test.shape)

(508, 5)
(128, 5)
